In [ ]:
# repo root + config (walk parents; do not use ../..)
from pathlib import Path
import json
import yaml

def _repo_root() -> Path:
    start = Path.cwd().resolve()
    for p in [start, *start.parents]:
        if (p / "config" / "config.yaml").is_file():
            return p
    raise FileNotFoundError("config/config.yaml not found walking from " + str(start))

PROJECT_ROOT = _repo_root()
CFG_YAML = yaml.safe_load((PROJECT_ROOT / "config" / "config.yaml").read_text(encoding="utf-8"))
_cfg_json = PROJECT_ROOT / "config" / "config.json"
if _cfg_json.is_file():
    with open(_cfg_json, encoding="utf-8") as _f:
        CFG = json.load(_f)



# RQ4: calculate the missing UMLS margins

The first margin pass **skipped** two cases. This notebook computes them with the
same PART2 SapBERT (mean-pool, L2) + FAISS index. Nothing is imputed.

**MedMentions encoders.** `output_text` is a CUI code, so it is not a valid query.
Query = `gold_mention`. \(s_1\) = max cosine of forms of the encoder's
`predicted_cui`; \(s_2\) = max cosine of a different CUI. Dummy `confidence=1.0`
is **not** used as \(s_1\).

**BioASQ / SQuAD2.** Query = predicted answer string. There is no assigned CUI,
so \(s_1\) = best CUI in the neighbourhood and \(s_2\) = best other CUI
(answer-text candidate margin). BERT/BioBERT/PubMedBERT still have no QA outputs
and cannot be calculated.

Generative MedMentions / all CADEC margins are left untouched.

In [ ]:
from pathlib import Path
from collections import defaultdict
import json
import gc
import shutil

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer
import faiss

PROJECT_ROOT = PROJECT_ROOT
assert (PROJECT_ROOT / "config" / "config.json").is_file()
assert torch.cuda.is_available(), "CUDA required — refuse CPU for SapBERT"

UNASSIGNED = "UNASSIGNED"
MIN_FORM_LEN = 3
FAISS_TOP_K = int(CFG_YAML["umls"]["faiss_top_k"])  # 1000 locked
_pool_config_name = f"sapbert_full_len{MIN_FORM_LEN}"
EMB_DIR = Path.home() / "data" / "umls" / "embeddings" / _pool_config_name
_SAP_SRC = (
    Path.home() / "data/hf_cache/hub"
    / "models--cambridgeltl--SapBERT-from-PubMedBERT-fulltext"
    / "snapshots" / "090663c3ae57bf35ffe4d0d468a2a88d03051a4d"
)

ENCODERS = ["BERT-base", "BioBERT", "PubMedBERT"]
MAPPED_PATH = PROJECT_ROOT / "outputs/rq1/intermediate/rq1_all_outputs_mapped.csv"
MM_MARGIN_PATH = PROJECT_ROOT / "outputs/rq1/umls_candidate_margin_medmentions.csv"
QA_PATH = PROJECT_ROOT / "outputs/qa/qa_results_combined.csv"
QA_MARGIN_PATH = PROJECT_ROOT / "outputs/qa/umls_candidate_margin_qa.csv"
OUT_DIR = PROJECT_ROOT / "outputs" / "rq4"
ALL_FIG = PROJECT_ROOT / "outputs" / "figures" / "all_rq_figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
ALL_FIG.mkdir(parents=True, exist_ok=True)

for p in (EMB_DIR / "surface_forms.json", EMB_DIR / "cui_form_pairs.json",
          EMB_DIR / "faiss.index", _SAP_SRC, MAPPED_PATH, MM_MARGIN_PATH, QA_PATH):
    assert Path(p).exists(), p

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Loading FAISS from {EMB_DIR}")
with open(EMB_DIR / "surface_forms.json", "r", encoding="utf-8") as f:
    _unique_forms = json.load(f)
with open(EMB_DIR / "cui_form_pairs.json", "r", encoding="utf-8") as f:
    _form_cui_pairs = [tuple(x) for x in json.load(f)]
_faiss_index = faiss.read_index(str(EMB_DIR / "faiss.index"))
assert len(_unique_forms) == _faiss_index.ntotal

def _norm_cui(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return UNASSIGNED
    s = str(x).strip()
    if s.startswith("UMLS:"):
        s = s[5:]
    if s in {"", "NA", "nan", "None", UNASSIGNED}:
        return UNASSIGNED
    return s

_form_to_cuis = defaultdict(set)
for _c, _f in _form_cui_pairs:
    _form_to_cuis[_f].add(_norm_cui(_c))

print(f"FAISS {_faiss_index.ntotal:,} forms")


In [ ]:
def _mean_pool(last_hidden, attn_mask):
    mask = attn_mask.unsqueeze(-1).expand(last_hidden.size()).float()
    summed = torch.sum(last_hidden * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts


def _embed_with_model(model, tokenizer, texts, batch_size=128, max_len=64, desc="sapbert"):
    vecs = []
    model.eval()
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc=desc, unit="batch"):
            batch = [str(t) if pd.notna(t) else "" for t in texts[i:i + batch_size]]
            enc = tokenizer(batch, return_tensors="pt", truncation=True,
                            max_length=max_len, padding=True)
            enc = {k: v.to("cuda", non_blocking=True) for k, v in enc.items()}
            out = model(**enc)
            pooled = _mean_pool(out.last_hidden_state, enc["attention_mask"])
            pooled = torch.nn.functional.normalize(pooled.float(), p=2, dim=1)
            vecs.append(pooled.detach().cpu().numpy().astype(np.float32))
    return np.vstack(vecs) if vecs else np.zeros((0, 768), dtype=np.float32)


def _s1_s2_from_hits(sims, idxs, pred_cui):
    """CUI-level s1 (pred) and s2 (best other) from one FAISS row."""
    s1 = None
    s2 = None
    for sim, idx in zip(sims, idxs):
        if int(idx) < 0:
            continue
        form = _unique_forms[int(idx)]
        if len(form) < MIN_FORM_LEN:
            continue
        cuis = _form_to_cuis.get(form, ())
        sc = float(sim)
        if pred_cui != UNASSIGNED and pred_cui in cuis:
            if s1 is None or sc > s1:
                s1 = sc
        if any(c != pred_cui and c != UNASSIGNED for c in cuis):
            if s2 is None or sc > s2:
                s2 = sc
    return s1, s2


def _top_two_cuis(sims, idxs):
    """Best CUI vs best other CUI (used when there is no assigned predicted_cui)."""
    best = {}
    for sim, idx in zip(sims, idxs):
        if int(idx) < 0:
            continue
        form = _unique_forms[int(idx)]
        if len(form) < MIN_FORM_LEN:
            continue
        sc = float(sim)
        for c in _form_to_cuis.get(form, ()):
            if c == UNASSIGNED:
                continue
            prev = best.get(c)
            if prev is None or sc > prev:
                best[c] = sc
    if not best:
        return UNASSIGNED, 0.0, 0.0
    ranked = sorted(best.items(), key=lambda kv: kv[1], reverse=True)
    c1, s1 = ranked[0]
    s2 = ranked[1][1] if len(ranked) > 1 else 0.0
    return c1, float(s1), float(s2)


print(f"Loading SapBERT from {_SAP_SRC}")
_sap_tok = AutoTokenizer.from_pretrained(str(_SAP_SRC))
_sap_mdl = AutoModel.from_pretrained(str(_SAP_SRC))
_sap_mdl = _sap_mdl.to("cuda").eval().half()
assert next(_sap_mdl.parameters()).device.type == "cuda"
print("SapBERT on", next(_sap_mdl.parameters()).device)


In [ ]:
# MedMentions encoders: query gold_mention, not the CUI string
# keep_default_na=False: instance mm_0046685 has the literal gold_mention 'NA' (the
# abbreviation in "total AgNOR area / nuclear area (TAA / NA)"), which the default reader
# turns into NaN and which fails the assertion below. 15 MedMentions instances are affected
# (12 'NA', 3 'null'). See docs/BUG_AUDIT.md. Applied 2026-09-14; this half is NOT executed
# before the 21 September cutoff, so the cutoff run picks the fix up rather than
# rediscovering the defect.
df_mapped = pd.read_csv(MAPPED_PATH, low_memory=False,
                        keep_default_na=False, na_values=[""])
df_mapped["predicted_cui"] = df_mapped["predicted_cui"].map(_norm_cui)
df_mapped["is_direct_cui"] = df_mapped["is_direct_cui"].astype(bool)
df_enc = df_mapped[df_mapped["model_name"].isin(ENCODERS)].copy()
assert df_enc["is_direct_cui"].all()
assert df_enc["gold_mention"].notna().all()
assert (df_enc["predicted_cui"] != UNASSIGNED).all()
print(f"encoder mapped rows: {len(df_enc):,}")

mentions = df_enc["gold_mention"].astype(str).tolist()
uniq_mentions = sorted(set(mentions))
print(f"unique gold_mention queries: {len(uniq_mentions):,}")
uniq_vecs = _embed_with_model(_sap_mdl, _sap_tok, uniq_mentions,
                              desc="encoder gold_mention")
D50, I50 = _faiss_index.search(uniq_vecs.astype(np.float32), FAISS_TOP_K)
mention_to_ui = {t: i for i, t in enumerate(uniq_mentions)}

# Exact s1: max SapBERT cosine of the predicted CUI's forms (not "0 if outside top-k")
print("Building CUI → form-index map for exact s1")
_form_index = {f: i for i, f in enumerate(_unique_forms)}
_cui_to_idxs = defaultdict(list)
for _c, _f in _form_cui_pairs:
    _cui_to_idxs[_norm_cui(_c)].append(_form_index[_f])
_emb = np.load(str(EMB_DIR / "embeddings.npy"), mmap_mode="r")

s1_cache = {}
s1_vals = np.empty(len(df_enc), dtype=np.float32)
s2_vals = np.empty(len(df_enc), dtype=np.float32)
preds = df_enc["predicted_cui"].tolist()
n_no_forms = 0
for i, (ment, pred) in enumerate(tqdm(list(zip(mentions, preds)), desc="encoder s1/s2")):
    ui = mention_to_ui[ment]
    _, s2 = _s1_s2_from_hits(D50[ui], I50[ui], pred)
    s2_vals[i] = 0.0 if s2 is None else s2
    key = (ui, pred)
    if key not in s1_cache:
        idxs = _cui_to_idxs.get(pred, [])
        if not idxs:
            s1_cache[key] = 0.0
            n_no_forms += 1
        else:
            vecs = np.asarray(_emb[idxs], dtype=np.float32)
            s1_cache[key] = float(np.max(uniq_vecs[ui] @ vecs.T))
    s1_vals[i] = s1_cache[key]
print(f"unique (mention, CUI) s1 lookups: {len(s1_cache):,}  CUIs with no forms: {n_no_forms}")
print(f"s1 mean={float(np.mean(s1_vals)):.4f}  s2 mean={float(np.mean(s2_vals)):.4f}")
assert float(np.mean(s1_vals)) > 0.05, "s1 collapsed — mention vs predicted-CUI cosine looks empty"

df_enc = df_enc.copy()
df_enc["s1"] = s1_vals
df_enc["s2"] = s2_vals
df_enc["umls_margin"] = df_enc["s1"] - df_enc["s2"]
assert df_enc["umls_margin"].notna().all()
print(df_enc.groupby("model_name")[["s1", "s2", "umls_margin"]].mean().round(4).to_string())
print(df_enc["umls_margin"].describe().round(4).to_string())

orig = (
    df_enc[df_enc["input_type"] == "original"]
    .drop_duplicates(["instance_id", "model_name"], keep="first")
    [["instance_id", "model_name", "umls_margin"]]
    .rename(columns={"umls_margin": "margin_original"})
)
df_margin = (
    df_enc.groupby(["instance_id", "model_name"], as_index=False)
    .agg(
        n_retrieval_rows=("umls_margin", "size"),
        margin_mean=("umls_margin", "mean"),
        margin_min=("umls_margin", "min"),
        s1_mean=("s1", "mean"),
        s2_mean=("s2", "mean"),
    )
    .merge(orig, on=["instance_id", "model_name"], how="left")
)
print("instance-level coverage:\n", df_margin.groupby("model_name").size().to_string())

bak = MM_MARGIN_PATH.with_suffix(".csv.bak_before_encoder_margin")
if not bak.exists():
    shutil.copy2(MM_MARGIN_PATH, bak)
    print("backup", bak)
merged = pd.read_csv(MM_MARGIN_PATH)
enc_mask = merged["model_name"].isin(ENCODERS)
n_enc_na = int(merged.loc[enc_mask, "margin_mean"].isna().sum())
n_gen_na = int(merged.loc[~enc_mask, "margin_mean"].isna().sum())
print(f"NA margin before fill: encoders={n_enc_na}  generatives={n_gen_na}")
assert n_enc_na == int(enc_mask.sum()), "encoder margin already filled — refusing overwrite"
assert n_gen_na <= 5, f"too many generative NA margins: {n_gen_na}"
gen_margin_before = merged.loc[~enc_mask, "margin_mean"].astype(float).to_numpy()

cols = ["n_retrieval_rows", "margin_original", "margin_mean", "margin_min", "s1_mean", "s2_mean"]
key = merged.loc[enc_mask, ["instance_id", "model_name"]].reset_index(drop=True)
filled = key.merge(df_margin, on=["instance_id", "model_name"], how="left")
assert filled["margin_mean"].notna().all(), "encoder merge left NA — instance_id mismatch"
assert len(filled) == int(enc_mask.sum())
for c in cols:
    merged.loc[enc_mask, c] = filled[c].to_numpy()

assert merged.loc[enc_mask, "margin_mean"].notna().all()
assert np.allclose(
    merged.loc[~enc_mask, "margin_mean"].astype(float).to_numpy(),
    gen_margin_before,
    equal_nan=True,
)
# WRITE MECHANICS ONLY -- nothing about what the bytes are changes here.
# This is the only read-modify-write onto a canonical artefact in the pipeline: it loads
# MM_MARGIN_PATH, adds encoder columns, and wrote back over the live path with a plain
# to_csv, so a kill mid-write left a TRUNCATED margin file. Temp-then-rename makes the
# final name appear only on completion, as everywhere else in the chain.
import os as _os11
_tmp11 = MM_MARGIN_PATH.with_suffix(MM_MARGIN_PATH.suffix + ".tmp")
merged.to_csv(_tmp11, index=False)
_os11.replace(_tmp11, MM_MARGIN_PATH)
print(f"Wrote {MM_MARGIN_PATH}  encoder margin defined: {int(enc_mask.sum()):,}")
print(merged.groupby("model_name")["margin_mean"].agg(n="size", defined="count", mean="mean").round(4).to_string())

H = merged["normalised_semantic_entropy_full"].astype(float).clip(lower=0)
zero = merged[(H == 0) & merged["model_name"].isin(ENCODERS)]
print(f"H=0 encoder rows: {len(zero):,}  margin std={zero['margin_mean'].std(ddof=1):.4f}")


In [ ]:
# QA: candidate margin of the predicted answer string
# keep_default_na=False for the same reason: one FLAN-T5-base SQuAD 2.0 prediction is the
# literal string 'None' and the default reader makes it NaN, which fillna("") then embeds as
# an empty string rather than as the answer the model gave.
qa = pd.read_csv(QA_PATH, keep_default_na=False, na_values=[""])
qa = qa[qa["m"] >= 3].copy()
qa["pred"] = qa["pred"].fillna("").astype(str)
print(f"QA included rows: {len(qa):,}  unique preds: {qa['pred'].nunique():,}")
print("QA models (encoders were never run):", sorted(qa.model.unique()))

uniq_pred = qa["pred"].unique().tolist()
pred_vecs = _embed_with_model(_sap_mdl, _sap_tok, uniq_pred, desc="qa pred")
Dq, Iq = _faiss_index.search(pred_vecs.astype(np.float32), FAISS_TOP_K)
pred_to_ui = {t: i for i, t in enumerate(uniq_pred)}

rows = []
for _, r in qa.iterrows():
    ui = pred_to_ui[r["pred"]]
    cui, s1, s2 = _top_two_cuis(Dq[ui], Iq[ui])
    rows.append({
        "id": r["id"], "model": r["model"], "dataset": r["dataset"],
        "pred": r["pred"], "correct": r["correct"],
        "m": r["m"], "norm_entropy": r["norm_entropy"],
        "confidence": r["confidence"],
        "assigned_cui": cui, "s1": s1, "s2": s2,
        "margin_mean": s1 - s2,
        "query": "pred_answer",
        "k": FAISS_TOP_K,
    })
qa_margin = pd.DataFrame(rows)
assert qa_margin["margin_mean"].notna().all()
QA_MARGIN_PATH.parent.mkdir(parents=True, exist_ok=True)
qa_margin.to_csv(QA_MARGIN_PATH, index=False)
print(f"Wrote {QA_MARGIN_PATH}  n={len(qa_margin):,}")
print(qa_margin.groupby(["dataset", "model"])["margin_mean"].agg(n="size", mean="mean", std="std").round(4).to_string())

# Cleanup only. The old form was `del _sap_mdl, uniq_vecs, pred_vecs, D50, I50, Dq, Iq`,
# which coupled this QA cell to cell 4: uniq_vecs/D50/I50 are cell 4 locals, so running
# the QA half alone raised NameError AFTER the CSV was written and failed the job under
# set -e. The two halves are independent populations (MedMentions encoders vs QA) and
# nothing but this line tied them together.
for _n in ("_sap_mdl", "uniq_vecs", "pred_vecs", "D50", "I50", "Dq", "Iq"):
    globals().pop(_n, None)
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# AURC / risk-coverage writes REMOVED.
# RQ4_margin_benchmark.ipynb is the sole writer of rq4_aurc_*,
# rq4_risk_coverage_*, rq4_combined3_wintest.csv, and rq4_aurc_bootstrap_ci.csv.
print("skip AURC/curve export — produced later by RQ4_margin_benchmark")
print(f"MM margin (encoder-filled): {MM_MARGIN_PATH}")
print(f"QA margin: {QA_MARGIN_PATH}")
